# 🎯 Phase 7: Milestone Exam Solutions

> **BI Analytics, Governance & Modern Data Stack**
>
> This notebook contains comprehensive solutions for all four Phase Milestone Exam questions.
> SQL queries use Python's `sqlite3` module for Pyodide compatibility.

---

## Question 1: Advanced SQL Analytics — Window Functions

**Combines**: Advanced SQL (Day 73), Data Preparation (Day 74)

**Scenario**: Write SQL queries using window functions to:
1. Calculate running totals and moving averages
2. Rank products by revenue within categories
3. Compute Year-over-Year growth rates
4. Identify churn using LAG/LEAD

### Theory: Window Functions

Window functions perform calculations **across rows related to the current row** without collapsing the result set (unlike GROUP BY).

```sql
function() OVER (
    PARTITION BY column    -- Groups (like GROUP BY but keeps all rows)
    ORDER BY column        -- Sort within partition
    ROWS/RANGE BETWEEN ... -- Frame (which rows to include)
)
```

| Function | Purpose | Example |
|----------|---------|--------|
| `ROW_NUMBER()` | Unique sequential number | Deduplicate records |
| `RANK()` | Rank with gaps for ties | Top-K rankings |
| `DENSE_RANK()` | Rank without gaps | Category rankings |
| `LAG(col, n)` | Previous row's value | Compare to prior period |
| `LEAD(col, n)` | Next row's value | Predict churn |
| `SUM() OVER` | Running total | Cumulative revenue |
| `AVG() OVER` | Moving average | Smoothed trends |

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cur = conn.cursor()

# Create and populate sales data
cur.executescript("""
    CREATE TABLE sales (
        id INTEGER PRIMARY KEY,
        sale_date TEXT,
        product TEXT,
        category TEXT,
        revenue REAL,
        quantity INTEGER
    );

    INSERT INTO sales (sale_date, product, category, revenue, quantity) VALUES
        ('2023-01-15', 'Widget A', 'Electronics', 1200, 10),
        ('2023-02-20', 'Widget A', 'Electronics', 1500, 12),
        ('2023-03-10', 'Widget A', 'Electronics', 1800, 15),
        ('2023-04-05', 'Widget A', 'Electronics', 1100, 9),
        ('2023-05-12', 'Widget A', 'Electronics', 2000, 16),
        ('2023-06-18', 'Widget A', 'Electronics', 2200, 18),
        ('2023-01-20', 'Gadget B', 'Electronics', 800, 5),
        ('2023-02-25', 'Gadget B', 'Electronics', 950, 6),
        ('2023-03-15', 'Gadget B', 'Electronics', 1100, 7),
        ('2023-04-10', 'Gadget B', 'Electronics', 750, 4),
        ('2023-05-20', 'Gadget B', 'Electronics', 1200, 8),
        ('2023-06-25', 'Gadget B', 'Electronics', 1400, 9),
        ('2023-01-10', 'Chair C', 'Furniture', 500, 2),
        ('2023-02-15', 'Chair C', 'Furniture', 750, 3),
        ('2023-03-20', 'Chair C', 'Furniture', 1000, 4),
        ('2023-04-25', 'Chair C', 'Furniture', 600, 2),
        ('2023-05-30', 'Chair C', 'Furniture', 1250, 5),
        ('2023-06-05', 'Chair C', 'Furniture', 900, 3),
        ('2023-01-12', 'Desk D', 'Furniture', 1500, 3),
        ('2023-02-18', 'Desk D', 'Furniture', 1800, 4),
        ('2023-03-22', 'Desk D', 'Furniture', 2100, 5),
        ('2023-04-28', 'Desk D', 'Furniture', 1200, 2),
        ('2023-05-15', 'Desk D', 'Furniture', 2400, 6),
        ('2023-06-20', 'Desk D', 'Furniture', 2700, 7);
""")
conn.commit()
print("✅ Sales data loaded (24 rows)")

In [ ]:
# 1. Running Total + Moving Average by Product
print("=" * 65)
print("QUERY 1: Running Total & Moving Average")
print("=" * 65)

query = """
    SELECT
        product,
        SUBSTR(sale_date, 1, 7) AS month,
        revenue,
        SUM(revenue) OVER (
            PARTITION BY product
            ORDER BY sale_date
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        ) AS running_total,
        ROUND(AVG(revenue) OVER (
            PARTITION BY product
            ORDER BY sale_date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ), 0) AS moving_avg_3
    FROM sales
    WHERE product IN ('Widget A', 'Desk D')
    ORDER BY product, sale_date;
"""

rows = cur.execute(query).fetchall()
print(f"{'Product':>10s} {'Month':>8s} {'Revenue':>9s} {'RunTotal':>10s} {'3-MA':>8s}")
print("-" * 50)
for p, m, r, rt, ma in rows:
    print(f"{p:>10s} {m:>8s} ${r:>7,.0f} ${rt:>8,.0f} ${ma:>6,.0f}")

In [ ]:
# 2. Rank Products by Total Revenue Within Categories
print("=" * 65)
print("QUERY 2: Product Rankings by Category")
print("=" * 65)

query = """
    WITH product_totals AS (
        SELECT
            category,
            product,
            SUM(revenue) AS total_revenue,
            SUM(quantity) AS total_qty
        FROM sales
        GROUP BY category, product
    )
    SELECT
        category,
        product,
        total_revenue,
        total_qty,
        RANK() OVER (PARTITION BY category ORDER BY total_revenue DESC) AS revenue_rank,
        ROUND(100.0 * total_revenue / SUM(total_revenue) OVER (PARTITION BY category), 1) AS pct_of_category
    FROM product_totals
    ORDER BY category, revenue_rank;
"""

rows = cur.execute(query).fetchall()
print(
    f"{'Category':>12s} {'Product':>10s} {'Revenue':>10s} {'Qty':>6s} {'Rank':>6s} {'% Cat':>7s}"
)
print("-" * 55)
for cat, p, rev, qty, rank, pct in rows:
    medal = {1: "🥇", 2: "🥈", 3: "🥉"}.get(rank, "  ")
    print(f"{cat:>12s} {p:>10s} ${rev:>8,.0f} {qty:>6d} {medal}{rank:<4d} {pct:>6.1f}%")

In [ ]:
# 3. Month-over-Month Growth with LAG
print("=" * 65)
print("QUERY 3: Month-over-Month Growth (LAG)")
print("=" * 65)

query = """
    WITH monthly AS (
        SELECT
            SUBSTR(sale_date, 1, 7) AS month,
            SUM(revenue) AS total_revenue
        FROM sales
        GROUP BY month
    )
    SELECT
        month,
        total_revenue,
        LAG(total_revenue, 1) OVER (ORDER BY month) AS prev_month,
        CASE
            WHEN LAG(total_revenue, 1) OVER (ORDER BY month) IS NOT NULL
            THEN ROUND(100.0 * (total_revenue - LAG(total_revenue, 1) OVER (ORDER BY month))
                 / LAG(total_revenue, 1) OVER (ORDER BY month), 1)
            ELSE NULL
        END AS growth_pct
    FROM monthly
    ORDER BY month;
"""

rows = cur.execute(query).fetchall()
print(f"{'Month':>8s} {'Revenue':>10s} {'Prev':>10s} {'Growth':>8s}")
print("-" * 40)
for m, rev, prev, growth in rows:
    prev_str = f"${prev:>8,.0f}" if prev else f"{'—':>9s}"
    growth_str = f"{growth:>+6.1f}%" if growth is not None else f"{'—':>7s}"
    trend = "📈" if growth and growth > 0 else "📉" if growth and growth < 0 else " "
    print(f"{m:>8s} ${rev:>8,.0f} {prev_str} {growth_str} {trend}")

---

## Question 2: Data Quality Framework

**Combines**: Data Governance (Day 79), Data Quality (Day 80)

**Scenario**: Implement automated data quality checks for the sales data:
1. Completeness (no NULLs in critical fields)
2. Uniqueness (no duplicate records)
3. Validity (values within acceptable ranges)
4. Consistency (referential integrity)
5. Timeliness (data freshness)

### Theory: Data Quality Dimensions

| Dimension | Definition | Example Check |
|-----------|-----------|---------------|
| **Completeness** | No missing values in required fields | `WHERE col IS NULL` |
| **Uniqueness** | No duplicate records | `GROUP BY ... HAVING COUNT(*) > 1` |
| **Validity** | Values within expected ranges | `WHERE price < 0 OR price > 999999` |
| **Consistency** | Cross-field and cross-table agreement | `revenue = price × quantity` |
| **Timeliness** | Data arrives within expected window | Last record < 24 hours old |

In [ ]:
def run_quality_checks(cursor, table="sales"):
    """
    Automated data quality framework.

    Returns a list of check results with pass/fail status.
    """
    results = []

    # 1. Completeness: No NULLs in critical fields
    for col in ["sale_date", "product", "category", "revenue"]:
        n = cursor.execute(
            f"SELECT COUNT(*) FROM {table} WHERE {col} IS NULL"
        ).fetchone()[0]
        results.append(
            {
                "dimension": "Completeness",
                "check": f"{col} not NULL",
                "passed": n == 0,
                "detail": f"{n} NULL values" if n > 0 else "All populated",
            }
        )

    # 2. Uniqueness: No exact duplicates
    n = cursor.execute(f"""
        SELECT COUNT(*) FROM (
            SELECT sale_date, product, revenue
            FROM {table}
            GROUP BY sale_date, product, revenue
            HAVING COUNT(*) > 1
        )
    """).fetchone()[0]
    results.append(
        {
            "dimension": "Uniqueness",
            "check": "No duplicate (date, product, revenue)",
            "passed": n == 0,
            "detail": f"{n} duplicate groups" if n > 0 else "No duplicates",
        }
    )

    # 3. Validity: Revenue > 0, Quantity > 0
    for col, min_val in [("revenue", 0), ("quantity", 0)]:
        n = cursor.execute(
            f"SELECT COUNT(*) FROM {table} WHERE {col} <= {min_val}"
        ).fetchone()[0]
        results.append(
            {
                "dimension": "Validity",
                "check": f"{col} > {min_val}",
                "passed": n == 0,
                "detail": f"{n} invalid values" if n > 0 else "All valid",
            }
        )

    # 4. Validity: Date format
    n = cursor.execute(f"""
        SELECT COUNT(*) FROM {table}
        WHERE sale_date NOT GLOB '[0-9][0-9][0-9][0-9]-[0-9][0-9]-[0-9][0-9]'
    """).fetchone()[0]
    results.append(
        {
            "dimension": "Validity",
            "check": "Valid date format (YYYY-MM-DD)",
            "passed": n == 0,
            "detail": f"{n} malformed dates" if n > 0 else "All valid",
        }
    )

    # 5. Consistency: Category matches product
    n = cursor.execute(f"""
        SELECT COUNT(*) FROM (
            SELECT product, COUNT(DISTINCT category) as cat_count
            FROM {table}
            GROUP BY product
            HAVING cat_count > 1
        )
    """).fetchone()[0]
    results.append(
        {
            "dimension": "Consistency",
            "check": "Product → Category mapping consistent",
            "passed": n == 0,
            "detail": f"{n} products with multiple categories"
            if n > 0
            else "Consistent",
        }
    )

    return results


# Run checks
print("=" * 55)
print("DATA QUALITY REPORT")
print("=" * 55)

checks = run_quality_checks(cur)
passed = sum(1 for c in checks if c["passed"])
total = len(checks)

print(f"\nOverall: {passed}/{total} checks passed\n")

for c in checks:
    icon = "✅" if c["passed"] else "❌"
    print(f"  {icon} [{c['dimension']:>13s}] {c['check']:40s} → {c['detail']}")

---

## Question 3: A/B Test Analysis

**Combines**: Experimentation (Day 78), Statistics (Day 76)

**Scenario**: Analyze an A/B test for a new checkout flow:
- Control: Existing 3-step checkout
- Treatment: New 1-step checkout
- Primary metric: Conversion rate
- Secondary metrics: Average order value, time to checkout

### Theory: Hypothesis Testing

- **H₀ (Null)**: New checkout has the same conversion rate
- **H₁ (Alt)**: New checkout has a different (higher) conversion rate
- **α = 0.05**: 5% significance level (1 in 20 chance of false positive)
- **p-value**: Probability of seeing this result if H₀ is true. If p < α → reject H₀.

In [ ]:
import numpy as np

# Simulate A/B test data
np.random.seed(42)

n_control = 5000
n_treatment = 5000

# True conversion rates
p_control = 0.032  # 3.2% conversion
p_treatment = 0.038  # 3.8% conversion (+18.75% lift)

control_conversions = np.random.binomial(1, p_control, n_control)
treatment_conversions = np.random.binomial(1, p_treatment, n_treatment)

# Calculate statistics
c_rate = control_conversions.mean()
t_rate = treatment_conversions.mean()
lift = (t_rate - c_rate) / c_rate * 100

# Two-proportion Z-test
p_pooled = (control_conversions.sum() + treatment_conversions.sum()) / (
    n_control + n_treatment
)
se = np.sqrt(p_pooled * (1 - p_pooled) * (1 / n_control + 1 / n_treatment))
z_stat = (t_rate - c_rate) / se


# Approximate p-value (one-tailed)
# Using normal approximation: P(Z > z)
def normal_cdf(x):
    """Approximate standard normal CDF using error function."""
    return 0.5 * (1 + np.tanh(np.sqrt(2 / np.pi) * (x + 0.044715 * x**3)))


p_value = 1 - normal_cdf(z_stat)

print("=" * 55)
print("A/B TEST RESULTS")
print("=" * 55)

print(f"\n{'Metric':>25s} {'Control':>12s} {'Treatment':>12s}")
print("-" * 52)
print(f"{'Sample Size':>25s} {n_control:>12,d} {n_treatment:>12,d}")
print(
    f"{'Conversions':>25s} {control_conversions.sum():>12d} {treatment_conversions.sum():>12d}"
)
print(f"{'Conversion Rate':>25s} {c_rate:>11.2%} {t_rate:>11.2%}")

print(f"\n{'Lift':>25s} {lift:>+.1f}%")
print(f"{'Z-statistic':>25s} {z_stat:>.3f}")
print(f"{'p-value (one-tailed)':>25s} {p_value:>.4f}")

sig = p_value < 0.05
print(f"\n{'Significant at α=0.05?':>25s} {'✅ YES' if sig else '❌ NO'}")

if sig:
    print("\n🎉 Result: The new checkout flow SIGNIFICANTLY increases")
    print(f"   conversion rate by {lift:+.1f}% (p={p_value:.4f}).")
    print("   Recommendation: Roll out to all users.")
else:
    print("\n⏳ Result: No significant difference detected.")
    print("   Recommendation: Continue test with larger sample size.")

In [ ]:
# Sample Size Calculator for future tests
def required_sample_size(baseline_rate, min_detectable_effect, alpha=0.05, power=0.80):
    """
    Calculate required sample size per group for a two-proportion test.

    Uses the formula: n = (Z_alpha + Z_beta)^2 * (p1(1-p1) + p2(1-p2)) / (p2-p1)^2
    """
    # Z-scores for common alpha and power values
    z_alpha = {0.05: 1.96, 0.01: 2.576, 0.10: 1.645}[alpha]
    z_beta = {0.80: 0.842, 0.90: 1.282, 0.95: 1.645}[power]

    p1 = baseline_rate
    p2 = baseline_rate * (1 + min_detectable_effect)

    numerator = (z_alpha + z_beta) ** 2 * (p1 * (1 - p1) + p2 * (1 - p2))
    denominator = (p2 - p1) ** 2

    return int(np.ceil(numerator / denominator))


print("\n" + "=" * 55)
print("SAMPLE SIZE CALCULATOR")
print("=" * 55)
print(f"\nBaseline conversion rate: {p_control:.1%}")
print(
    f"\n{'MDE (lift)':>15s} {'n per group':>12s} {'Total':>10s} {'Days @ 1K/day':>15s}"
)
print("-" * 55)

for mde in [0.05, 0.10, 0.20, 0.30, 0.50]:
    n = required_sample_size(p_control, mde)
    days = int(np.ceil(n * 2 / 1000))
    print(f"{mde:>14.0%} {n:>12,d} {n * 2:>10,d} {days:>15d}")

print("\n💡 Smaller effects need exponentially larger samples to detect.")
print("   This is why A/B tests take weeks — you need statistical power.")

---

## Question 4: Data Storytelling — Dashboard Design

**Combines**: Storytelling (Day 78), Visualization (Day 75-76), BI Platforms (Day 77)

**Scenario**: Design a data storytelling presentation for the C-suite:
1. Structure the narrative (situation → complication → resolution)
2. Choose appropriate chart types
3. Calculate the KPIs that matter

### Theory: The SCR Framework

The **Situation-Complication-Resolution** (SCR) framework structures data stories:

| Element | Purpose | Example |
|---------|---------|--------|
| **Situation** | Context (what's normal) | "Our MRR grew 15% in Q1..." |
| **Complication** | The problem/opportunity | "...but churn doubled in the SMB segment" |
| **Resolution** | Data-driven recommendation | "Feature X reduces churn by 40% — prioritize it for SMB" |

### Chart Selection Guide

| What to Show | Best Chart | Why |
|--------------|-----------|-----|
| Trend over time | Line chart | Shows direction and pattern |
| Comparison | Horizontal bar | Easy to compare magnitudes |
| Composition | Stacked bar or pie | Shows parts of a whole |
| Distribution | Histogram or box plot | Shows spread and outliers |
| Correlation | Scatter plot | Shows relationship strength |

In [ ]:
# Build the executive KPI dashboard data
import sqlite3

conn2 = sqlite3.connect(":memory:")
cur2 = conn2.cursor()

cur2.executescript("""
    CREATE TABLE kpi_monthly (
        month TEXT, segment TEXT,
        mrr REAL, new_customers INTEGER, churned INTEGER,
        nps INTEGER, support_tickets INTEGER
    );
    INSERT INTO kpi_monthly VALUES
        ('2024-01', 'Enterprise', 450000, 5, 1, 72, 45),
        ('2024-01', 'SMB', 180000, 25, 8, 55, 210),
        ('2024-02', 'Enterprise', 470000, 6, 0, 74, 38),
        ('2024-02', 'SMB', 175000, 20, 12, 48, 280),
        ('2024-03', 'Enterprise', 495000, 8, 1, 76, 32),
        ('2024-03', 'SMB', 165000, 18, 15, 42, 350);
""")
conn2.commit()

# Build the SCR narrative
print("=" * 60)
print("EXECUTIVE DASHBOARD — Q1 2024 REVIEW")
print("=" * 60)

# SITUATION
total_mrr = cur2.execute("""
    SELECT month, SUM(mrr) FROM kpi_monthly GROUP BY month ORDER BY month
""").fetchall()
print("\n📌 SITUATION:")
print(f"   Total MRR grew from ${total_mrr[0][1]:,.0f} to ${total_mrr[-1][1]:,.0f}")
growth = (total_mrr[-1][1] - total_mrr[0][1]) / total_mrr[0][1]
print(f"   Q1 growth: {growth:+.1%}")

# COMPLICATION
smb_data = cur2.execute("""
    SELECT month, mrr, churned, nps, support_tickets
    FROM kpi_monthly WHERE segment = 'SMB' ORDER BY month
""").fetchall()
print("\n⚠️ COMPLICATION:")
print(f"   SMB segment MRR declined: ${smb_data[0][1]:,.0f} → ${smb_data[-1][1]:,.0f}")
print(f"   SMB churn tripled: {smb_data[0][2]} → {smb_data[-1][2]} customers")
print(f"   SMB NPS collapsed: {smb_data[0][3]} → {smb_data[-1][3]}")
print(f"   SMB support tickets surged: {smb_data[0][4]} → {smb_data[-1][4]}")

# RESOLUTION
print("\n✅ RESOLUTION:")
print("   1. Root cause: Recent UI change increased SMB time-to-value by 3x")
print("   2. Action: Revert UI for SMB tier, keep for Enterprise")
print("   3. Impact: Expected to recover $15K MRR/month within 60 days")
print("   4. Prevention: Segment A/B tests by customer tier going forward")

# Segment comparison table
print(f"\n{'─' * 60}")
print(f"{'SEGMENT COMPARISON':^60s}")
print(f"{'─' * 60}")

segments = cur2.execute("""
    SELECT segment,
           SUM(mrr) / 3 AS avg_mrr,
           SUM(new_customers) AS total_new,
           SUM(churned) AS total_churned,
           ROUND(AVG(nps), 0) AS avg_nps
    FROM kpi_monthly GROUP BY segment
""").fetchall()

print(
    f"{'Segment':>12s} {'Avg MRR':>12s} {'New':>6s} {'Churn':>6s} {'NPS':>6s} {'Health':>8s}"
)
for seg, mrr, new, churn, nps in segments:
    health = "🟢" if nps > 60 else "🟡" if nps > 50 else "🔴"
    print(f"{seg:>12s} ${mrr:>10,.0f} {new:>6d} {churn:>6d} {nps:>6.0f} {health:>8s}")

conn2.close()

---

## 🎓 Summary

This notebook demonstrated solutions to all four Phase 7 Milestone Exam questions:

1. **Advanced SQL**: Window functions (running totals, rankings, LAG/LEAD for MoM growth)
2. **Data Quality Framework**: Automated checks for completeness, uniqueness, validity, consistency
3. **A/B Test Analysis**: Two-proportion Z-test, p-value, sample size calculator
4. **Data Storytelling**: SCR framework, KPI dashboard, segment analysis

Phase 7 bridges analytical skills and business communication — knowing the SQL is not enough; you must **tell the story** the data reveals.